# California house pricing — the project at a glance

Instant median-value estimates for California census block groups, served by
a **synchronous SageMaker endpoint with a Feast online store (DynamoDB on
moto)** — the deployment shape for interactive, per-request scoring.

This collection follows the same production lifecycle as the NYC taxi demand
project: frame the decision, collect data, engineer features into a store,
train, tune, evaluate, deploy, monitor — every stage a runnable notebook.

## The lifecycle

| Stage | Notebook | What it does |
|---|---|---|
| Frame | `docs/01_problem_framing.md` | Why sync endpoint + online store |
| Collect | `docs/02_data_collection.md` | California census dataset |
| Preprocess | `docs/03_data_preprocessing.md` | Entity keys + Feast tables |
| Explore | `eda.ipynb` | What drives house values |
| Feature store | `feature_engineering.ipynb` | Feast apply + materialize to DynamoDB |
| Train | `training.ipynb` | LightGBM job on moto S3 |
| Tune | `tuning.ipynb` | num_leaves / learning_rate sweep |
| Evaluate | `evaluation.ipynb` | Holdout R²/MAE via inference image |
| Deploy | `deploy.ipynb` | Endpoint + online feature lookup + invoke |
| Monitor | `monitoring.ipynb` | Endpoint stats + teardown |

## Architecture

```
prepare_data ──> feature_repo (Feast)
                    │ offline (duckdb)          │ online (dynamodb via moto)
                    ▼                           ▼
        get_historical_features ──> train job ──> model ──> endpoint <── get_online_features
```

The endpoint serves one block group per call; the features it scores come
from the same Feast repo the training set was built from — one definition,
offline and online.